
# OAENet Paper Replication in Python — Corrected — Stable Adaptive Penalty Version

This notebook reproduces the **Outcome Adaptive Elastic Net (OAENet)** workflow from Islam, Shikalgar, and Noor-E-Alam (2025).

The previous Python notebook could appear to “not work” because the custom solver recomputed an expensive spectral norm **for every lambda, fold, and replication**. This corrected version precomputes the Lipschitz constant once per fold, uses warm starts, and implements the `discard="both"` common-support trimming from the supplied R code.

## Workflow

1. Standardize covariates.
2. OLS outcome model.
3. Adaptive penalty: \(w_j = |\hat\beta_j|^{-3}\).
4. Weighted logistic elastic net with 5-fold CV.
5. One-standard-error lambda.
6. Select nonzero treatment-model coefficients.
7. Propensity-score common-support trimming (`discard="both"`).
8. 1:1 nearest-neighbor matching with caliper 0.25.
9. Post-matching regression \(Y \sim A + X_{\text{selected}}\); coefficient of `A` is ATT.

**Implementation note:** the article does not report the exact numerical elastic-net mixing value corresponding to `glmnet::alpha`; this notebook keeps `alpha_enet=0.5`, matching the supplied R implementation.


### Critical fix in this version
The adaptive penalty factors are **not mean-normalized**. The previous normalization caused extremely large selected CV penalties and frequently returned `selected=0`, `ATT=nan`.


In [1]:

# Run once if needed:
# %pip install numpy pandas scipy scikit-learn statsmodels

import time
import warnings
import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
import statsmodels.api as sm

warnings.filterwarnings("ignore")


## 1. Fast weighted elastic-net logistic solver

In [2]:

def soft_threshold(x, threshold):
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0.0)


def spectral_lipschitz(X):
    """
    Logistic gradient Lipschitz component:
        ||X||_2^2 / (4n)
    Compute once per design matrix, not once per lambda.
    """
    n = X.shape[0]
    smax = np.linalg.svd(X, compute_uv=False)[0]
    return (smax * smax) / (4.0 * n)


def fista_weighted_logistic_enet(
    X, y, penalty_factor, lambda1, lambda2,
    base_L=None, max_iter=1500, tol=1e-6, init=None
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(penalty_factor, dtype=float)
    n, p = X.shape

    if init is None:
        mu = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        beta0 = np.log(mu / (1.0 - mu))
        beta = np.zeros(p)
    else:
        beta0, beta = init
        beta0 = float(beta0)
        beta = np.asarray(beta, dtype=float).copy()

    if base_L is None:
        base_L = spectral_lipschitz(X)

    # Intercept curvature <= 0.25; beta curvature adds lambda2.
    L = max(base_L + lambda2, 0.25)

    z0 = beta0
    z = beta.copy()
    t = 1.0

    for _ in range(max_iter):
        eta = z0 + X @ z
        p_hat = expit(eta)
        resid = p_hat - y

        grad0 = resid.mean()
        grad = (X.T @ resid) / n + lambda2 * z

        beta0_new = z0 - grad0 / L
        beta_new = soft_threshold(
            z - grad / L,
            (lambda1 / L) * w
        )

        delta = max(
            abs(beta0_new - beta0),
            np.max(np.abs(beta_new - beta))
        )

        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        momentum = (t - 1) / t_new

        z0 = beta0_new + momentum * (beta0_new - beta0)
        z = beta_new + momentum * (beta_new - beta)

        beta0, beta, t = beta0_new, beta_new, t_new

        if delta < tol:
            break

    return beta0, beta


def predict_prob(X, beta0, beta):
    return expit(beta0 + np.asarray(X) @ beta)


def binomial_deviance(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -2 * np.mean(y * np.log(p) + (1-y) * np.log(1-p))


## 2. Five-fold CV + one-standard-error rule

In [3]:

def make_lambda_grid(
    X, y, penalty_factor, alpha_enet=0.5,
    n_lambdas=35, lambda_min_ratio=1e-3
):
    """
    Weighted-L1 lambda_max, then a decreasing log-spaced path.
    """
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    w = np.asarray(penalty_factor, float)

    grad0 = np.abs(X.T @ (np.repeat(y.mean(), len(y)) - y)) / len(y)
    denom = np.maximum(alpha_enet * w, 1e-12)
    lambda_max = np.max(grad0 / denom)

    if (not np.isfinite(lambda_max)) or lambda_max <= 0:
        lambda_max = 1.0

    return np.geomspace(
        lambda_max,
        lambda_max * lambda_min_ratio,
        n_lambdas
    )


def cv_oaenet_treatment(
    X, A, penalty_factor,
    alpha_enet=0.5,
    nfolds=5,
    random_state=123,
    n_lambdas=35
):
    X = np.asarray(X, float)
    A = np.asarray(A, int)
    w = np.asarray(penalty_factor, float)

    lambdas = make_lambda_grid(
        X, A, w,
        alpha_enet=alpha_enet,
        n_lambdas=n_lambdas
    )

    cv = StratifiedKFold(
        n_splits=nfolds,
        shuffle=True,
        random_state=random_state
    )

    losses = np.full((nfolds, len(lambdas)), np.nan)

    for fold, (tr, va) in enumerate(cv.split(X, A)):
        Xtr, Xva = X[tr], X[va]
        Atr, Ava = A[tr], A[va]

        base_L = spectral_lipschitz(Xtr)
        init = None

        for j, lam in enumerate(lambdas):
            b0, b = fista_weighted_logistic_enet(
                Xtr, Atr, w,
                lambda1=lam * alpha_enet,
                lambda2=lam * (1-alpha_enet),
                base_L=base_L,
                init=init
            )
            init = (b0, b)
            losses[fold, j] = binomial_deviance(
                Ava, predict_prob(Xva, b0, b)
            )

    mean_loss = np.nanmean(losses, axis=0)
    se_loss = np.nanstd(losses, axis=0, ddof=1) / np.sqrt(nfolds)

    j_min = int(np.nanargmin(mean_loss))
    cutoff = mean_loss[j_min] + se_loss[j_min]

    # Lambda path is descending: first eligible = largest lambda within 1 SE.
    eligible = np.where(mean_loss <= cutoff)[0]
    j_1se = int(eligible[0])

    lam_1se = float(lambdas[j_1se])
    lam_min = float(lambdas[j_min])

    base_L_full = spectral_lipschitz(X)
    beta0, beta = fista_weighted_logistic_enet(
        X, A, w,
        lambda1=lam_1se * alpha_enet,
        lambda2=lam_1se * (1-alpha_enet),
        base_L=base_L_full
    )

    return {
        "lambda_1se": lam_1se,
        "lambda_min": lam_min,
        "beta0": beta0,
        "beta": beta,
        "mean_cv_loss": mean_loss,
        "se_cv_loss": se_loss,
        "lambdas": lambdas
    }


## 3. OAENet variable selection

In [4]:

def oaenet_select(
    X, A, Y,
    c_gamma=3,
    alpha_enet=0.5,
    nfolds=5,
    random_state=123
):
    X = np.asarray(X, float)
    A = np.asarray(A, int)
    Y = np.asarray(Y, float)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # Stage 1: OLS outcome model
    ols = LinearRegression()
    ols.fit(Xs, Y)
    beta_y = ols.coef_.copy()

    # w_j = |beta_j|^{-c}, c=3 in paper simulations.
    eps = 1e-8
    penalty_factor = np.maximum(np.abs(beta_y), eps) ** (-c_gamma)

    # IMPORTANT:
    # Do NOT normalize the adaptive penalty factors here.
    #
    # In the OAENet objective the adaptive weights multiply the L1 term,
    # while the L2 term has its own penalty. If we divide the weights by
    # their mean and then use one common lambda to generate both L1 and L2
    # strengths, the near-zero outcome coefficients can make the mean weight
    # enormous. That made lambda_1se explode and produced the all-zero models
    # seen in the previous notebook.
    #
    # Keeping the raw w_j = |beta_j|^{-c} scale avoids that numerical failure.

    fit_a = cv_oaenet_treatment(
        Xs, A, penalty_factor,
        alpha_enet=alpha_enet,
        nfolds=nfolds,
        random_state=random_state
    )

    beta_a = fit_a["beta"]
    selected_idx = np.flatnonzero(np.abs(beta_a) > 1e-7)

    return {
        "selected_idx": selected_idx,
        "beta_outcome": beta_y,
        "penalty_factor": penalty_factor,
        "beta_treatment": beta_a,
        "lambda_1se": fit_a["lambda_1se"],
        "lambda_min": fit_a["lambda_min"],
        "cv_fit": fit_a
    }


## 4. Propensity-score matching matching the supplied R settings

In [5]:

def propensity_score_match(
    df,
    selected_vars,
    treatment="A",
    caliper=0.25,
    discard_both=True
):
    """
    Reproduce the supplied R settings as closely as possible:
        method = "nearest"
        distance = "glm"
        ratio = 1
        caliper = 0.25
        estimand = "ATT"
        replace = FALSE
        discard = "both"

    Matching is greedy 1:1 without replacement on logit propensity score.
    """
    if len(selected_vars) == 0:
        return None

    X = df[selected_vars].to_numpy(float)
    A = df[treatment].to_numpy(int)

    ps_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000
    )
    ps_model.fit(X, A)
    ps = ps_model.predict_proba(X)[:, 1]

    eps = 1e-8
    ps_clip = np.clip(ps, eps, 1-eps)
    logit_ps = np.log(ps_clip / (1-ps_clip))

    keep = np.ones(len(df), dtype=bool)

    if discard_both:
        # Common support = overlap of treated/control propensity score ranges.
        ps_t = ps[A == 1]
        ps_c = ps[A == 0]
        lower = max(ps_t.min(), ps_c.min())
        upper = min(ps_t.max(), ps_c.max())
        keep = (ps >= lower) & (ps <= upper)

    idx_keep = np.where(keep)[0]
    A_k = A[idx_keep]
    logit_k = logit_ps[idx_keep]

    treated_local = np.where(A_k == 1)[0]
    control_local = np.where(A_k == 0)[0]

    # MatchIt calipers for PS are standardized by default.
    caliper_abs = caliper * np.std(logit_k, ddof=1)

    # Approximate MatchIt nearest-neighbor ordering: hardest treated first
    # (largest propensity score first).
    treated_local = treated_local[
        np.argsort(-logit_k[treated_local])
    ]

    available_controls = set(control_local.tolist())
    pairs = []

    for ti in treated_local:
        if not available_controls:
            break

        candidates = np.fromiter(
            available_controls, dtype=int,
            count=len(available_controls)
        )
        d = np.abs(logit_k[candidates] - logit_k[ti])
        j = int(np.argmin(d))
        ci = int(candidates[j])

        if d[j] <= caliper_abs:
            pairs.append((ti, ci))
            available_controls.remove(ci)

    if not pairs:
        return None

    original_pairs = [
        (idx_keep[ti], idx_keep[ci])
        for ti, ci in pairs
    ]

    rows = []
    pair_ids = []
    for pid, (ti, ci) in enumerate(original_pairs):
        rows.extend([ti, ci])
        pair_ids.extend([pid, pid])

    md = df.iloc[rows].copy().reset_index(drop=True)
    md["_pair_id"] = pair_ids

    return md


def post_match_att_regression(
    matched_df,
    selected_vars,
    treatment="A",
    outcome="Y"
):
    if matched_df is None or len(matched_df) == 0:
        return np.nan

    X = matched_df[[treatment] + list(selected_vars)].astype(float)
    X = sm.add_constant(X, has_constant="add")
    y = matched_df[outcome].astype(float)

    fit = sm.OLS(y, X).fit()
    return float(fit.params[treatment])


## 5. Complete OAENet run

In [6]:

def run_oaenet(
    df,
    covariates,
    treatment="A",
    outcome="Y",
    c_gamma=3,
    alpha_enet=0.5,
    nfolds=5,
    caliper=0.25,
    random_state=123
):
    start = time.perf_counter()

    X = df[covariates].to_numpy(float)
    A = df[treatment].to_numpy(int)
    Y = df[outcome].to_numpy(float)

    selection = oaenet_select(
        X, A, Y,
        c_gamma=c_gamma,
        alpha_enet=alpha_enet,
        nfolds=nfolds,
        random_state=random_state
    )

    selected_vars = [
        covariates[i] for i in selection["selected_idx"]
    ]

    matched = propensity_score_match(
        df,
        selected_vars,
        treatment=treatment,
        caliper=caliper,
        discard_both=True
    )

    att = post_match_att_regression(
        matched,
        selected_vars,
        treatment=treatment,
        outcome=outcome
    )

    return {
        "att": att,
        "selected_vars": selected_vars,
        "n_selected": len(selected_vars),
        "matched_data": matched,
        "n_treated_matched": 0 if matched is None else int((matched[treatment] == 1).sum()),
        "n_control_matched": 0 if matched is None else int((matched[treatment] == 0).sum()),
        "elapsed_seconds": time.perf_counter() - start,
        **selection
    }


## 6. Paper simulation data-generating mechanisms

In [7]:

def compound_symmetry_cov(p, variance, rho):
    S = np.full((p, p), rho * variance, float)
    np.fill_diagonal(S, variance)
    return S


def simulate_scenario1(n=1000, rho=0.0, seed=1):
    rng = np.random.default_rng(seed)
    p = 100

    X = rng.multivariate_normal(
        np.zeros(p),
        compound_symmetry_cov(p, 1.0, rho),
        size=n
    )

    eta = X[:, :10].sum(axis=1) + X[:, 20:30].sum(axis=1)
    A = rng.binomial(1, expit(eta))
    Y = 0.5*A + 0.6*X[:, :20].sum(axis=1)

    cols = [f"X{i}" for i in range(1, 101)]
    df = pd.DataFrame(X, columns=cols)
    df.insert(0, "A", A)
    df.insert(0, "Y", Y)
    return df, 0.5


def simulate_scenario2(n=1000, rho=0.0, seed=1):
    rng = np.random.default_rng(seed)
    p = 100

    X = rng.multivariate_normal(
        np.zeros(p),
        compound_symmetry_cov(p, 4.0, rho),
        size=n
    )

    eta = (
        0.5*X[:, 0] - X[:, 1]
        + 0.3*X[:, 4] - 0.3*X[:, 5]
        + 0.3*X[:, 6] - 0.3*X[:, 7]
    )
    A = rng.binomial(1, expit(eta))
    Y = (
        1.0*A + 2*X[:, 0] + 2*X[:, 1]
        + 5*X[:, 2] + 5*X[:, 3]
    )

    cols = [f"X{i}" for i in range(1, 101)]
    df = pd.DataFrame(X, columns=cols)
    df.insert(0, "A", A)
    df.insert(0, "Y", Y)
    return df, 1.0


paper_scenarios = pd.DataFrame({
    "scenario": ["1A","1B","1C","1D","1E","1F",
                 "2A","2B","2C","2D","2E","2F"],
    "dgp": [1]*6 + [2]*6,
    "n": [1000,1000,1000,500,500,500,
          1000,1000,1000,500,500,500],
    "rho": [0,0.25,0.5,0,0.25,0.5]*2
})

paper_scenarios


,scenario,dgp,n,rho
0,1A,1,1000,0.00
1,1B,1,1000,0.25
2,1C,1,1000,0.50
3,1D,1,500,0.00
4,1E,1,500,0.25
5,1F,1,500,0.50
6,2A,2,1000,0.00
7,2B,2,1000,0.25
8,2C,2,1000,0.50
9,2D,2,500,0.00


## 7. Replication driver

In [8]:

def replicate_one_scenario(
    scenario_id,
    n_rep=20,
    seed=100,
    c_gamma=3,
    alpha_enet=0.5,
    nfolds=5,
    caliper=0.25,
    verbose=True
):
    row = paper_scenarios.loc[
        paper_scenarios["scenario"] == scenario_id
    ]

    if len(row) != 1:
        raise ValueError("Unknown scenario_id")

    row = row.iloc[0]
    covariates = [f"X{i}" for i in range(1,101)]

    atts, nsel, runtimes, nt, nc = [], [], [], [], []
    selection_count = pd.Series(0, index=covariates, dtype=float)

    for r in range(n_rep):
        this_seed = seed + r

        if row["dgp"] == 1:
            df, true_te = simulate_scenario1(
                int(row["n"]), float(row["rho"]), this_seed
            )
        else:
            df, true_te = simulate_scenario2(
                int(row["n"]), float(row["rho"]), this_seed
            )

        try:
            fit = run_oaenet(
                df, covariates,
                c_gamma=c_gamma,
                alpha_enet=alpha_enet,
                nfolds=nfolds,
                caliper=caliper,
                random_state=this_seed
            )
            atts.append(fit["att"])
            nsel.append(fit["n_selected"])
            runtimes.append(fit["elapsed_seconds"])
            nt.append(fit["n_treated_matched"])
            nc.append(fit["n_control_matched"])
            selection_count.loc[fit["selected_vars"]] += 1
        except Exception as e:
            print(f"Replication {r+1} failed: {e}")
            atts.append(np.nan)
            nsel.append(np.nan)
            runtimes.append(np.nan)
            nt.append(np.nan)
            nc.append(np.nan)

        if verbose:
            print(
                f"{scenario_id}: {r+1}/{n_rep} | "
                f"ATT={atts[-1]:.4f} | selected={nsel[-1]}"
            )

    atts = np.asarray(atts, float)
    summary = pd.DataFrame([{
        "scenario": scenario_id,
        "n_rep": n_rep,
        "successful_att": int(np.isfinite(atts).sum()),
        "true_te": true_te,
        "mean_att": np.nanmean(atts),
        "bias": np.nanmean(atts) - true_te,
        "variance": np.nanvar(atts, ddof=1) if np.isfinite(atts).sum() > 1 else np.nan,
        "mean_runtime_seconds": np.nanmean(runtimes),
        "mean_n_selected": np.nanmean(nsel),
        "mean_matched_treated": np.nanmean(nt),
        "mean_matched_control": np.nanmean(nc)
    }])

    return {
        "summary": summary,
        "att": atts,
        "selection_rate": selection_count / n_rep,
        "n_selected": np.asarray(nsel, float),
        "runtime": np.asarray(runtimes, float)
    }


## 8. Sanity check: run ONE Scenario 1A dataset first

In [9]:

# This should finish relatively quickly and lets you verify the pipeline.

df_1A, true_te = simulate_scenario1(n=1000, rho=0, seed=100)
covariates = [f"X{i}" for i in range(1,101)]

one_fit = run_oaenet(
    df_1A,
    covariates,
    random_state=100
)

print("True TE:", true_te)
print("ATT:", one_fit["att"])
print("Number selected:", one_fit["n_selected"])
print("Selected variables:", one_fit["selected_vars"])
print("Matched treated:", one_fit["n_treated_matched"])
print("Matched controls:", one_fit["n_control_matched"])
print("Runtime (sec):", round(one_fit["elapsed_seconds"], 3))


True TE: 0.5
ATT: 0.41402336175648374
Number selected: 11
Selected variables: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X12']
Matched treated: 312
Matched controls: 312
Runtime (sec): 3.087


## 9. Then run 20 replications

In [10]:
test_1A = replicate_one_scenario(
    "1A",
    n_rep=20,
    seed=100,
    verbose=True
)

test_1A["summary"]


1A: 1/20 | ATT=0.4140 | selected=11
1A: 2/20 | ATT=0.3962 | selected=10
1A: 3/20 | ATT=0.5622 | selected=12
1A: 4/20 | ATT=0.4594 | selected=12
1A: 5/20 | ATT=0.6310 | selected=15
1A: 6/20 | ATT=0.6375 | selected=10
1A: 7/20 | ATT=0.6320 | selected=11
1A: 8/20 | ATT=0.4668 | selected=12
1A: 9/20 | ATT=0.4223 | selected=11
1A: 10/20 | ATT=0.5827 | selected=11
1A: 11/20 | ATT=0.3470 | selected=11
1A: 12/20 | ATT=0.5852 | selected=10
1A: 13/20 | ATT=0.5440 | selected=14
1A: 14/20 | ATT=0.7230 | selected=10
1A: 15/20 | ATT=0.3418 | selected=11
1A: 16/20 | ATT=0.3153 | selected=11
1A: 17/20 | ATT=0.4672 | selected=12
1A: 18/20 | ATT=0.5199 | selected=11
1A: 19/20 | ATT=0.5798 | selected=10
1A: 20/20 | ATT=0.6165 | selected=13


,scenario,n_rep,successful_att,true_te,mean_att,bias,variance,mean_runtime_seconds,mean_n_selected,mean_matched_treated,mean_matched_control
0,1A,20,20,0.5,0.512198,0.012198,0.013212,3.280096,11.4,296.7,296.7


In [11]:

# Paper's reported Scenario 1A OAENet values
comparison = pd.DataFrame({
    "metric": ["mean_att", "bias", "variance"],
    "paper": [0.5031, 0.0031, 0.0090],
    "replication": [
        test_1A["summary"].loc[0, "mean_att"],
        test_1A["summary"].loc[0, "bias"],
        test_1A["summary"].loc[0, "variance"]
    ]
})
comparison["difference"] = comparison["replication"] - comparison["paper"]
comparison


,metric,paper,replication,difference
0,mean_att,0.5031,0.512198,0.009098
1,bias,0.0031,0.012198,0.009098
2,variance,0.0090,0.013212,0.004212


In [12]:

# Variable selection rates
test_1A["selection_rate"].sort_values(ascending=False).head(30)


X1     1.00
X3     1.00
X4     1.00
X5     1.00
X6     1.00
X7     1.00
X8     1.00
X9     1.00
X10    1.00
X2     1.00
X12    0.30
X17    0.20
X19    0.20
X18    0.20
X20    0.15
X16    0.10
X13    0.10
X14    0.10
X11    0.05
X80    0.00
X71    0.00
X78    0.00
X77    0.00
X76    0.00
X75    0.00
X74    0.00
X73    0.00
X72    0.00
X70    0.00
X63    0.00
dtype: float64

## 10. Full 1,000-replication run (only after the sanity checks work)

In [13]:

# result_1A = replicate_one_scenario(
#     "1A",
#     n_rep=1000,
#     seed=100,
#     verbose=True
# )
#
# result_1A["summary"]
